# Evaluación Parcial N°3 – Car Seats

**Docente**: Mariela Moraga

**Ramo**: Programación para la Ciencia de Datos (SCY1101)  

**Sección**: 002D

**Estudiante**: Milenka Guerra

**Dataset**: `dataset_car_seats.csv`

---

## 0. Importaciones y Configuración Global

Se importan las librerías necesarias para todas las etapas de este proyecto.

In [7]:
# Manejo de datos
import os

os.environ["PYTHONWARNINGS"] = "ignore"

import pandas as pd
import numpy as np

# Consumo de API
import requests

# Evitar warnings en funciones obsoletas
import warnings

warnings.filterwarnings("ignore")

# Configuraciones globales
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Pipeline ETL

El pipeline integra dos fuentes de datos: el archivo `dataset_car_seats.csv` (datos originales de ventas de sillas para niños) y una API REST de tipo de cambio de divisa, usada para convertir las columnas monetarias de dólares (USD) a pesos chilenos (CLP). Sobre esa base se aplica la conversión USD→CLP y la creación de la variable `HighSales`, dejando el resultado validado y exportado en `data/processed/` como primer paso.

### 1.1 Carga de Datos desde CSV

In [8]:
RAW_PATH = "../data/raw/dataset_car_seats.csv"


def cargar_dataset(ruta):
    try:
        return pd.read_csv(ruta, index_col=0)
    except FileNotFoundError:
        raise FileNotFoundError(f"No se encontró el archivo en la ruta: {ruta}")


df = cargar_dataset(RAW_PATH)

print(f"Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()

Dimensiones: 400 filas y 11 columnas.


,Sales,CompPrice,Income,Advertising,Population,Price,ShelveLoc,Age,Education,Urban,US
0,9.50,138,73,11,276,120,Bad,42,17,Yes,Yes
1,11.22,111,48,16,260,83,Good,65,10,Yes,Yes
2,10.06,113,35,10,269,80,Medium,59,12,Yes,Yes
3,7.40,117,100,4,466,97,Medium,55,14,Yes,Yes
4,4.15,141,64,3,340,128,Bad,38,13,Yes,No


### 1.2 Estructura e Integridad del Dataset Original

In [9]:
# Revisión de la estructura del DataFrame
tabla_estructura = pd.DataFrame(
    {
        "Tipo de Dato": df.dtypes.astype(str),
        "Valores Únicos": [df[col].nunique() for col in df.columns],
        "Datos Nulos (NaN)": df.isnull().sum(),
    }
)

display(tabla_estructura)
print(f"Registros duplicados: {df.duplicated().sum()}")

,Tipo de Dato,Valores Únicos,Datos Nulos (NaN)
Sales,float64,336,0
CompPrice,int64,73,0
Income,int64,98,0
Advertising,int64,28,0
Population,int64,275,0
Price,int64,101,0
ShelveLoc,str,3,0
Age,int64,56,0
Education,int64,9,0
Urban,str,2,0


Registros duplicados: 0


El dataset está completo: no presenta valores nulos ni registros duplicados, por lo que no se requiere imputación previa a las transformaciones.

- `ShelveLoc`, `Urban` y `US` son las únicas columnas categóricas (`str`), con 3, 2 y 2 niveles respectivamente.
- El resto de las columnas son numéricas (`int64`/`float64`), sin columnas constantes ni identificadores a eliminar.

### 1.3 Identificación de Columnas Monetarias

Según la descripción oficial de las columnas, cuatro de ellas están expresadas en dólares y deben convertirse a CLP:

| Columna | Descripción | ¿Monetaria? |
|-|-|-|
| `CompPrice` | Precio que cobra el competidor en esa ubicación | Sí (USD) |
| `Income` | Nivel de ingreso de la comunidad (en miles de dólares) | Sí (USD, en miles) |
| `Advertising` | Presupuesto local de publicidad de la empresa (en miles de dólares) | Sí (USD, en miles) |
| `Price` | Precio que cobra la empresa por las sillas en esa tienda | Sí (USD) |
| `Sales` | Ventas de sillas **en miles de unidades** | No, es una cantidad de unidades, no dinero |
| `Population` | Tamaño de la población de la región **en miles** | No, es un conteo de personas |

Se excluyen `Sales` y `Population` porque, aunque numéricas, no representan montos en dólares. `Income` y `Advertising` sí son monetarias, pero ya están expresadas en miles de USD; al convertirlas, el resultado queda en miles de CLP (misma escala, otra moneda), mientras que `CompPrice` y `Price` quedan directamente en CLP.

In [10]:
COLUMNAS_MONETARIAS = ["CompPrice", "Income", "Advertising", "Price"]

print(f"Columnas monetarias a convertir: {COLUMNAS_MONETARIAS}")

Columnas monetarias a convertir: ['CompPrice', 'Income', 'Advertising', 'Price']


### 1.4 Obtención del Tipo de Cambio (API)

Se probaron las dos APIs sugeridas en el enunciado antes de implementar el código: `dolarapi.com` respondió de forma rápida y con un formato simple (`compra/venta`), mientras que `mindicador.cl/api/dolar` no respondió dentro del tiempo de espera en las pruebas realizadas. Por eso `dolarapi.com` se usa como fuente principal.

Aun así, se implementa un **fallback en cadena**, como medida de precaución. Si `dolarapi.com` falla, se intenta `mindicador.cl` como respaldo. En condiciones normales, `mindicador.cl` no llega a usarse porque la fuente principal responde correctamente, pero el código queda preparado para ese escenario.

Además, si por algún motivo ambas conexiones llegasen a fallar, el valor se obtiene de una variable fija que se congeló con el precio del dólar a día del 23 de junio de 2026. Como medida de resguardo para la evaluación.

De todas formas, al tener conexión fructífera con dolarapi, se usa el campo `venta` (valor al que se vende el dólar) como tipo de cambio de referencia para la conversión USD→CLP.

In [11]:
def obtener_dolarapi():
    url = "https://cl.dolarapi.com/v1/cotizaciones/usd"
    respuesta = requests.get(url, timeout=8)
    respuesta.raise_for_status()
    datos = respuesta.json()
    return float(datos["venta"])


def obtener_mindicador():
    url = "https://mindicador.cl/api/dolar"
    respuesta = requests.get(url, timeout=8)
    respuesta.raise_for_status()
    datos = respuesta.json()
    return float(datos["serie"][0]["valor"])


def obtener_tipo_cambio():
    global fuente_origen
    fuentes = [
        ("dolarapi.com", obtener_dolarapi),
        ("mindicador.cl", obtener_mindicador),
    ]

    for nombre_fuente, funcion in fuentes:
        try:
            valor = funcion()
            if valor is None or valor <= 0:
                raise ValueError(f"valor inválido recibido: {valor}")
            fuente_origen = nombre_fuente
            print(f"Tipo de cambio obtenido desde {nombre_fuente}: ${valor:,.2f} CLP")
            return valor
        except (
            requests.exceptions.RequestException,
            KeyError,
            ValueError,
            TypeError,
        ) as error:
            print(
                f"No se pudo obtener el tipo de cambio desde {nombre_fuente}: {error}"
            )

    # Fallback valor fijo: dolar al 23-jun-2026
    fuente_origen = "valor_fijo"
    valor_fijo = 911.46
    print(
        f"No se pudo obtener el tipo de cambio desde ninguna fuente. Usando valor fijo: ${valor_fijo:,.2f} CLP"
    )
    return valor_fijo


fuente_origen = None
tipo_cambio = obtener_tipo_cambio()
print(f"Fuente utilizada: {fuente_origen}")

Tipo de cambio obtenido desde dolarapi.com: $914.75 CLP
Fuente utilizada: dolarapi.com


### 1.5 Conversión USD → CLP

Las columnas monetarias se convierten reemplazando directamente el valor en dólares por su equivalente en pesos chilenos, redondeado a peso entero. Tras este paso, `CompPrice`, `Income`, `Advertising` y `Price` quedan expresadas en CLP. El formato con separador de miles (`123,456`) se aplica solo como presentación de la tabla; el dato subyacente se mantiene numérico (`int64`) para no afectar el EDA ni el modelado posteriores.

In [12]:
for columna in COLUMNAS_MONETARIAS:
    df[columna] = (df[columna] * tipo_cambio).round(0).astype(int)

df[COLUMNAS_MONETARIAS].head().style.format("{:,.0f}")

,CompPrice,Income,Advertising,Price
0,"126,236","66,777","10,062","109,770"
1,"101,537","43,908","14,636","75,924"
2,"103,367","32,016","9,148","73,180"
3,"107,026","91,475","3,659","88,731"
4,"128,980","58,544","2,744","117,088"


### 1.6 Creación de Variable HighSales

Se define `HighSales` como la variable objetivo para el modelado supervisado posterior, con un 1 si las ventas de la tienda (`Sales`, en miles de unidades) son superiores a 8, y 0 en caso contrario.

In [13]:
df["HighSales"] = (df["Sales"] > 8).astype(int)

conteo = df["HighSales"].value_counts()
proporciones = df["HighSales"].value_counts(normalize=True) * 100

resumen_highsales = pd.DataFrame(
    {"Cantidad": conteo, "Porcentaje (%)": proporciones.round(1)}
)
resumen_highsales.index = ["Bajas (0)", "Altas (1)"]

display(resumen_highsales)

,Cantidad,Porcentaje (%)
Bajas (0),236,59.00
Altas (1),164,41.00


La distribución resultante es 236 tiendas con ventas bajas (59.0%) y 164 con ventas altas (41.0%). Aquí las clases están razonablemente equilibradas, por lo que en el modelado posterior el *accuracy* sí será una métrica informativa, aunque conviene complementarla con F1 y ROC-AUC.

### 1.7 Validación de Datos Transformados

Antes de exportar, se verifica que las transformaciones no hayan introducido tipos de datos inesperados ni valores nulos.

In [14]:
tabla_validacion = pd.DataFrame(
    {
        "Tipo de Dato": df[COLUMNAS_MONETARIAS + ["HighSales"]].dtypes.astype(str),
        "Datos Nulos (NaN)": df[COLUMNAS_MONETARIAS + ["HighSales"]].isnull().sum(),
    }
)

display(tabla_validacion)
print(
    f"Columnas totales tras transformación: {df.shape[1]} (11 originales + HighSales; las 4 columnas monetarias se sobrescribieron en CLP)"
)
print(f"Registros duplicados: {df.duplicated().sum()}")

,Tipo de Dato,Datos Nulos (NaN)
CompPrice,int64,0
Income,int64,0
Advertising,int64,0
Price,int64,0
HighSales,int64,0


Columnas totales tras transformación: 12 (11 originales + HighSales; las 4 columnas monetarias se sobrescribieron en CLP)
Registros duplicados: 0


Las 4 columnas monetarias (`CompPrice`, `Income`, `Advertising`, `Price`) se mantienen como `int64`, ya que se redondean a peso entero. `HighSales` también queda como `int64` (0/1). No se introducen nulos. El dataset pasa de 11 a 12 columnas: se agrega únicamente `HighSales`, mientras que las columnas monetarias se sobrescriben en lugar de duplicarse. Por lo tanto, este dataset queda listo para el EDA y la preparación de los modelos.

### 1.8 Exportación del Dataset Limpio

In [15]:
PROCESSED_PATH = "../data/processed/dataset_etl.csv"

df.to_csv(PROCESSED_PATH, index=False)

print(f"Dataset ETL exportado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Ruta: {PROCESSED_PATH}")

Dataset ETL exportado: 400 filas x 12 columnas
Ruta: ../data/processed/dataset_etl.csv
